# Blinkit Sales Data — Cleaning & Feature Engineering

**Project:** Blinkit Sales Performance Dashboard
**Phase:** 2 of the pipeline — Python data cleaning & feature engineering
**Author:** Harsh Chauhan

**Pipeline:** Raw CSVs (Kaggle) → **this notebook (Python)** → SQL database → Power BI dashboard

This notebook takes the 9 raw Blinkit CSV files, profiles them for data quality issues,
resolves the ones found, engineers analysis-ready features, and exports clean CSVs
that will be loaded into a SQL database in Phase 3.


## 1. Setup & Load

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

customers   = pd.read_csv("blinkit_customers.csv", dtype={"phone": str, "pincode": str})
feedback    = pd.read_csv("blinkit_customer_feedback.csv")
delivery    = pd.read_csv("blinkit_delivery_performance.csv")
inventory   = pd.read_csv("blinkit_inventory.csv")
marketing   = pd.read_csv("blinkit_marketing_performance.csv")
orders      = pd.read_csv("blinkit_orders.csv")
order_items = pd.read_csv("blinkit_order_items.csv")
products    = pd.read_csv("blinkit_products.csv")

tables = {
    "customers": customers, "feedback": feedback, "delivery": delivery,
    "inventory": inventory, "marketing": marketing, "orders": orders,
    "order_items": order_items, "products": products,
}

for name, df in tables.items():
    print(f"{name:12s} -> {df.shape[0]:>6} rows, {df.shape[1]:>2} cols")


customers    ->   2500 rows, 11 cols
feedback     ->   5000 rows,  8 cols
delivery     ->   5000 rows,  8 cols
inventory    ->  75172 rows,  4 cols
marketing    ->   5400 rows, 11 cols
orders       ->   5000 rows, 10 cols
order_items  ->   5000 rows,  4 cols
products     ->    268 rows, 10 cols


**Note on `dtype` above:** `phone` and `pincode` are forced to load as text
(`dtype=str`), not inferred as numbers. Without this, pandas silently reads
`+912987579691` as the integer `912987579691` and drops the leading `+` country
code — a real data-loss bug that only shows up once you look closely at the
values, since the column still "looks numeric."


## 2. Data Quality Profiling

Checking each table for nulls, duplicate rows, and referential integrity
(do foreign keys actually match up across tables?) before touching anything.


In [2]:
for name, df in tables.items():
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    dupes = df.duplicated().sum()
    print(f"--- {name} ---")
    print("nulls:", dict(nulls) if len(nulls) else "none")
    print("duplicate rows:", dupes)
    print()


--- customers ---
nulls: none
duplicate rows: 0

--- feedback ---
nulls: none
duplicate rows: 0

--- delivery ---
nulls: {'reasons_if_delayed': np.int64(1902)}
duplicate rows: 0

--- inventory ---
nulls: none
duplicate rows: 0

--- marketing ---
nulls: none
duplicate rows: 0

--- orders ---
nulls: none
duplicate rows: 0

--- order_items ---
nulls: none
duplicate rows: 0

--- products ---
nulls: none
duplicate rows: 0



**Findings:**
- Every table is null-free **except** `delivery.reasons_if_delayed` — and that's expected,
  not messy: it's blank whenever a delivery was on time, so there's nothing to explain.
  This will be kept as `"Not Delayed"` rather than dropped or imputed.
- No duplicate rows in any of the 8 core tables.


In [3]:
for tbl, left, lkey, right, rkey in [
    ("orders -> customers",      orders, "customer_id",  customers, "customer_id"),
    ("order_items -> orders",    order_items, "order_id", orders, "order_id"),
    ("order_items -> products",  order_items, "product_id", products, "product_id"),
    ("delivery -> orders",       delivery, "order_id",   orders, "order_id"),
    ("feedback -> orders",       feedback, "order_id",   orders, "order_id"),
    ("feedback -> customers",    feedback, "customer_id", customers, "customer_id"),
]:
    match_rate = left[lkey].isin(right[rkey]).mean()
    print(f"{tbl:28s} match rate: {match_rate:.1%}")


orders -> customers          match rate: 100.0%
order_items -> orders        match rate: 100.0%
order_items -> products      match rate: 100.0%
delivery -> orders           match rate: 100.0%
feedback -> orders           match rate: 100.0%
feedback -> customers        match rate: 100.0%


**Finding:** every foreign key relationship matches at 100% — this dataset has
clean referential integrity, so no orphaned records need to be dropped.


## 3. Resolving the Inventory File Duplication

The raw data included two inventory files: `blinkit_inventory.csv` and
`blinkit_inventoryNew.csv`. Before picking one, compare them directly.


In [4]:
inventory_new = pd.read_csv("blinkit_inventoryNew.csv")

print("inventory.csv     :", inventory.shape, "| unique dates:", inventory['date'].nunique(),
      "| duplicate rows:", inventory.duplicated().sum())
print("inventoryNew.csv   :", inventory_new.shape, "| unique dates:", inventory_new['date'].nunique(),
      "| duplicate rows:", inventory_new.duplicated().sum())
print("\ninventory.csv date sample:    ", inventory['date'].head(2).tolist())
print("inventoryNew.csv date sample: ", inventory_new['date'].head(2).tolist())


inventory.csv     : (75172, 4) | unique dates: 600 | duplicate rows: 0
inventoryNew.csv   : (18105, 4) | unique dates: 21 | duplicate rows: 7359

inventory.csv date sample:     ['17-03-2023', '17-03-2023']
inventoryNew.csv date sample:  ['Mar-23', 'Mar-23']


**Decision: keep `inventory.csv`, drop `inventoryNew.csv`.**

`inventoryNew.csv` is a coarser, monthly-grain rollup (only 21 unique month labels
like `"Mar-23"`) with **7,359 duplicate rows** — it looks like an aggregated/rolled-up
export, not a corrected or more recent version. `inventory.csv` has genuine daily
granularity (600 unique dates) and zero duplicates, so it's the more useful and
trustworthy source for a day-level dashboard. `inventoryNew.csv` is excluded from
the rest of this pipeline.


## 4. Standardizing Date Formats

Each table uses a different date format, so each needs its own explicit parse
rather than a one-size-fits-all `pd.to_datetime` call.


In [5]:
orders['order_date']              = pd.to_datetime(orders['order_date'])
orders['promised_delivery_time']  = pd.to_datetime(orders['promised_delivery_time'])
orders['actual_delivery_time']    = pd.to_datetime(orders['actual_delivery_time'])

# inventory.csv uses DD-MM-YYYY (day-first) -- must set dayfirst=True or pandas
# will silently misparse day/month for anything <= day 12
inventory['date'] = pd.to_datetime(inventory['date'], dayfirst=True)

customers['registration_date'] = pd.to_datetime(customers['registration_date'])
feedback['feedback_date']      = pd.to_datetime(feedback['feedback_date'])
marketing['date']              = pd.to_datetime(marketing['date'])

delivery['promised_time'] = pd.to_datetime(delivery['promised_time'])
delivery['actual_time']   = pd.to_datetime(delivery['actual_time'])

print("All date columns parsed. Sample dtypes:")
print(orders[['order_date','promised_delivery_time','actual_delivery_time']].dtypes)
print(inventory['date'].dtype, "| inventory date range:", inventory['date'].min(), "to", inventory['date'].max())


All date columns parsed. Sample dtypes:
order_date                datetime64[us]
promised_delivery_time    datetime64[us]
actual_delivery_time      datetime64[us]
dtype: object
datetime64[us] | inventory date range: 2023-03-17 00:00:00 to 2024-11-05 00:00:00


## 5. Removing Redundant Columns Between `orders` and `delivery`

`orders` and `delivery` both carry `delivery_status`, promised time, and actual time
for the same `order_id` — confirmed identical above. To avoid duplicate logic in the
SQL model, `orders` stays the single source of truth for delivery status/timing, and
`delivery` is trimmed down to the columns that are genuinely unique to it.


In [6]:
status_match = (orders.set_index('order_id')['delivery_status']
                == delivery.set_index('order_id')['delivery_status']).mean()
print(f"orders vs delivery status agreement: {status_match:.1%}")

delivery_clean = delivery[[
    'order_id', 'delivery_partner_id', 'delivery_time_minutes',
    'distance_km', 'reasons_if_delayed'
]].copy()

delivery_clean['reasons_if_delayed'] = delivery_clean['reasons_if_delayed'].fillna('Not Delayed')
delivery_clean.head()


orders vs delivery status agreement: 100.0%


,order_id,delivery_partner_id,delivery_time_minutes,distance_km,reasons_if_delayed
0,1961864118,63230,-5.0,0.96,Not Delayed
1,1549769649,14983,2.0,0.98,Traffic
2,9185164487,39859,4.0,3.83,Traffic
3,9644738826,61497,-1.0,2.76,Not Delayed
4,5427684290,84315,1.0,2.63,Traffic


## 6. Fixing the Signed `delivery_time_minutes` Column

`delivery_time_minutes` is `actual_time − promised_time`, so negative values mean the
order arrived **early**, not that the data is broken. Rather than deleting or
`abs()`-ing this away, it's split into an explicit early/on-time/late flag plus a
non-negative delay magnitude for anyone building a "how late" metric later.


In [7]:
print("Negative (early) deliveries:", (delivery_clean['delivery_time_minutes'] < 0).sum())
print(delivery_clean['delivery_time_minutes'].describe())

def timing_flag(mins):
    if mins < 0:
        return "Early"
    elif mins == 0:
        return "On Time"
    else:
        return "Late"

delivery_clean['delivery_timing'] = delivery_clean['delivery_time_minutes'].apply(timing_flag)
delivery_clean['delay_minutes_abs'] = delivery_clean['delivery_time_minutes'].clip(lower=0)

delivery_clean['delivery_timing'].value_counts()


Negative (early) deliveries: 1563
count    5000.000000
mean        4.443000
std         8.063929
min        -5.000000
25%        -1.000000
50%         2.000000
75%         8.000000
max        30.000000
Name: delivery_time_minutes, dtype: float64


delivery_timing
Late       3098
Early      1563
On Time     339
Name: count, dtype: int64

## 7. Feature Engineering — Orders

Time-based and delivery-performance features derived from `orders`, ready for
month-over-month trend and SLA analysis in Power BI.


In [8]:
orders['order_month']      = orders['order_date'].dt.to_period('M').astype(str)
orders['order_day_of_week']= orders['order_date'].dt.day_name()
orders['order_hour']       = orders['order_date'].dt.hour

orders['promised_delivery_minutes'] = (
    (orders['promised_delivery_time'] - orders['order_date']).dt.total_seconds() / 60
)
orders['actual_delivery_minutes'] = (
    (orders['actual_delivery_time'] - orders['order_date']).dt.total_seconds() / 60
)

def delivery_bucket(mins):
    if mins < 10:
        return "<10 min"
    elif mins < 20:
        return "10-20 min"
    elif mins < 30:
        return "20-30 min"
    else:
        return "30+ min"

orders['delivery_time_bucket'] = orders['actual_delivery_minutes'].apply(delivery_bucket)

orders[['order_id','order_month','order_day_of_week','order_hour',
        'actual_delivery_minutes','delivery_time_bucket']].head()


,order_id,order_month,order_day_of_week,order_hour,actual_delivery_minutes,delivery_time_bucket
0,1961864118,2024-07,Wednesday,8,13.0,10-20 min
1,1549769649,2024-05,Tuesday,13,13.0,10-20 min
2,9185164487,2024-09,Monday,13,22.0,20-30 min
3,9644738826,2023-11,Friday,16,17.0,10-20 min
4,5427684290,2023-11,Monday,5,18.0,10-20 min


## 8. Feature Engineering — Products

Discount percentage from the `price` vs `mrp` gap, useful for margin/discount analysis
alongside the existing `margin_percentage` column.


In [9]:
products['discount_pct'] = (
    (products['mrp'] - products['price']) / products['mrp'] * 100
).round(2)

products[['product_name','category','mrp','price','discount_pct','margin_percentage']].head()


,product_name,category,mrp,price,discount_pct,margin_percentage
0,Onions,Fruits & Vegetables,1263.93,947.95,25.0,25.0
1,Potatoes,Fruits & Vegetables,169.55,127.16,25.0,25.0
2,Potatoes,Fruits & Vegetables,282.85,212.14,25.0,25.0
3,Tomatoes,Fruits & Vegetables,279.45,209.59,25.0,25.0
4,Onions,Fruits & Vegetables,472.69,354.52,25.0,25.0


## 9. Fixing Embedded Line Breaks in `customers.address`

`customers.address` contains genuine multi-line addresses (a literal newline character
inside the field, inside quotes) for about 75% of rows. This is valid CSV — pandas
reads it correctly — but tools with weaker CSV parsers (including MySQL Workbench's
Table Data Import Wizard) can misread these rows, since a stray line break makes a
single record look like two. Since address text doesn't need internal line breaks,
the newline is collapsed to a comma+space so every row is guaranteed to stay on one
physical line in the exported file.


In [10]:
embedded_newlines = customers['address'].astype(str).str.contains('\n').sum()
print(f"Addresses with an embedded line break: {embedded_newlines} of {len(customers)}")

customers['address'] = customers['address'].astype(str).str.replace('\n', ', ', regex=False)

print("After fix:", customers['address'].astype(str).str.contains('\n').sum(), "remaining")
customers['address'].head(3)


Addresses with an embedded line break: 1894 of 2500
After fix: 0 remaining


0    23, Nayar Path, Bihar Sharif-154625
1    51/302, Buch Chowk, Srinagar-570271
2     941, Anne Street, Darbhanga 186125
Name: address, dtype: str

## 10. Feature Engineering — Customers

A repeat-customer flag built from actual order counts in `orders`, cross-checked
against the `total_orders` column that already ships with `customers.csv`.


In [11]:
order_counts = orders.groupby('customer_id').size().rename('computed_order_count')
customers = customers.merge(order_counts, on='customer_id', how='left')
customers['computed_order_count'] = customers['computed_order_count'].fillna(0).astype(int)

customers['is_repeat_customer'] = customers['computed_order_count'] > 1

mismatch = (customers['computed_order_count'] != customers['total_orders']).mean()
print(f"Rows where provided total_orders != orders in this dataset: {mismatch:.1%}")
print("(Expected: total_orders likely reflects the customer's full lifetime history,")
print(" while this dataset only contains a sample of their orders.)")

customers[['customer_id','total_orders','computed_order_count','is_repeat_customer']].head()


Rows where provided total_orders != orders in this dataset: 95.4%
(Expected: total_orders likely reflects the customer's full lifetime history,
 while this dataset only contains a sample of their orders.)


,customer_id,total_orders,computed_order_count,is_repeat_customer
0,97475543,13,4,True
1,22077605,4,2,True
2,47822591,17,1,False
3,79726146,4,1,False
4,57102800,14,3,True


## 11. Final Null / Shape Check Before Export


In [12]:
final_tables = {
    "customers": customers,
    "feedback": feedback,
    "delivery": delivery_clean,
    "inventory": inventory,
    "marketing": marketing,
    "orders": orders,
    "order_items": order_items,
    "products": products,
}

for name, df in final_tables.items():
    print(f"{name:12s} -> {df.shape[0]:>6} rows, {df.shape[1]:>2} cols, "
          f"nulls: {df.isnull().sum().sum()}")


customers    ->   2500 rows, 13 cols, nulls: 0
feedback     ->   5000 rows,  8 cols, nulls: 0
delivery     ->   5000 rows,  7 cols, nulls: 0
inventory    ->  75172 rows,  4 cols, nulls: 0
marketing    ->   5400 rows, 11 cols, nulls: 0
orders       ->   5000 rows, 16 cols, nulls: 0
order_items  ->   5000 rows,  4 cols, nulls: 0
products     ->    268 rows, 11 cols, nulls: 0


## 12. Export Cleaned Tables

Exported as `*_cleaned.csv` — these are what get loaded into the SQL database in
Phase 3.


In [13]:
import os
os.makedirs("cleaned", exist_ok=True)

for name, df in final_tables.items():
    df.to_csv(f"cleaned/blinkit_{name}_cleaned.csv", index=False)

print("Exported:", os.listdir("cleaned"))


Exported: ['blinkit_marketing_cleaned.csv', 'blinkit_delivery_cleaned.csv', 'blinkit_orders_cleaned.csv', 'blinkit_feedback_cleaned.csv', 'blinkit_products_cleaned.csv', 'blinkit_inventory_cleaned.csv', 'blinkit_order_items_cleaned.csv', 'blinkit_customers_cleaned.csv']


## Summary

| Issue found | Resolution |
|---|---|
| `phone`/`pincode` silently read as numbers, dropping the `+91` prefix | Forced to load as text with `dtype=str` |
| Two inventory files, one with duplicates & coarser grain | Kept `inventory.csv`, dropped `inventoryNew.csv` |
| Mixed date formats across files (`DD-MM-YYYY` vs ISO) | Parsed each explicitly, `dayfirst=True` for inventory |
| `orders` and `delivery` duplicate delivery-status/timing columns | `orders` kept as source of truth; `delivery` trimmed to partner/distance/delay-reason only |
| Signed `delivery_time_minutes` (negative = early) | Split into `delivery_timing` flag + non-negative `delay_minutes_abs` |
| Blank `reasons_if_delayed` for on-time orders | Filled with `"Not Delayed"` rather than left null |
| Embedded line breaks inside `customers.address` (75% of rows) | Collapsed to `", "` so every record is a single physical CSV line — needed for reliable import into MySQL/Power BI |
| No time-based or discount features | Engineered `order_month`, `order_hour`, `delivery_time_bucket`, `discount_pct`, `is_repeat_customer` |

**Next phase:** load `cleaned/*.csv` into a SQL database and write the analytical
queries (top products, revenue trends, delivery SLA breakdowns, repeat customer
behavior) that will feed the Power BI model.
